In [ ]:
# ============================================================
# CELL 1: Libraries & Config
# ============================================================
import os
import re
import time
import requests
import pandas as pd
from getpass import getuser

USER = getuser()
OUTPUT_DIR = f"C:/Users/{USER}/Documents/GitHub/tennis-homophily/data/atp"
os.makedirs(OUTPUT_DIR, exist_ok=True)

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
DELAY = 1.5   # polite delay (seconds) between Wikipedia API calls

# Wikipedia's API policy requires a descriptive User-Agent.
# Requests without one (or with the default python-requests UA) get 403 Forbidden.
HEADERS = {
    "User-Agent": (
        "tennis-homophily-scraper/1.0 "
        "(academic research project; alessandrodimattia21@gmail.com) "
        "python-requests"
    )
}

# (wikipedia_page_title, tournament_label, year_label)
# En dash \u2013 is used in Wikipedia article titles.
TOURNAMENTS = [
    # --- 2024 Grand Slams ---
    ("2024 Australian Open \u2013 Men's doubles",          "Australian Open", "2024"),
    ("2024 French Open \u2013 Men's doubles",              "Roland Garros",   "2024"),
    ("2024 Wimbledon Championships \u2013 Men's doubles",  "Wimbledon",       "2024"),
    ("2024 US Open \u2013 Men's doubles",                  "US Open",         "2024"),
    # --- 2025 Grand Slams ---
    ("2025 Australian Open \u2013 Men's doubles",          "Australian Open", "2025"),
    ("2025 French Open \u2013 Men's doubles",              "Roland Garros",   "2025"),
    ("2025 Wimbledon Championships \u2013 Men's doubles",  "Wimbledon",       "2025"),
    ("2025 US Open \u2013 Men's doubles",                  "US Open",         "2025"),
    # --- Last two Olympics editions ---
    ("Tennis at the 2020 Summer Olympics \u2013 Men's doubles", "Olympics", "2021"),
    ("Tennis at the 2024 Summer Olympics \u2013 Men's doubles", "Olympics", "2024"),
]

# Map Wikipedia bracket round labels → clean stage names
STAGE_NORMALIZE = {
    "first round":               "First Round",
    "second round":              "Second Round",
    "third round":               "Third Round",
    "quarterfinals":             "Quarter-Finals",
    "quarter-finals":            "Quarter-Finals",
    "quarterfinal":              "Quarter-Finals",
    "quarter-final":             "Quarter-Finals",
    "semifinals":                "Semi-Finals",
    "semi-finals":               "Semi-Finals",
    "semifinal":                 "Semi-Finals",
    "semi-final":                "Semi-Finals",
    "final":                     "Final",
    "finals":                    "Final",
    # Olympics-specific
    "gold medal match":          "Final",
    "silver medal match":        "Final",
    "bronze medal match":        "Bronze Medal Match",
    "bronze medal play-off":     "Bronze Medal Match",
    "bronze medal play-offs":    "Bronze Medal Match",
    "first round (play-offs)":   "First Round",
    "second round (play-offs)":  "Second Round",
}

ROUND_ORDER = {
    "First Round":       1,
    "Second Round":      2,
    "Third Round":       3,
    "Quarter-Finals":    4,
    "Semi-Finals":       5,
    "Bronze Medal Match":5,
    "Final":             6,
}

print("Config loaded. Output dir:", OUTPUT_DIR)

In [ ]:
# ============================================================
# CELL 2: Wikipedia API — fetch raw wikitext
# ============================================================
def fetch_wikitext(page_title: str) -> str:
    """
    Fetch raw wikitext for a Wikipedia page via the MediaWiki API.
    Raises ValueError if the page is not found or the API returns an error.
    """
    resp = requests.get(
        WIKIPEDIA_API,
        headers=HEADERS,
        params={
            "action":        "parse",
            "page":          page_title,
            "prop":          "wikitext",
            "format":        "json",
            "formatversion": "2",
        },
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    if "error" in data:
        raise ValueError(f"API error for '{page_title}': {data['error'].get('info', data['error'])}")
    return data["parse"]["wikitext"]

In [ ]:
# ============================================================
# CELL 3: Template extraction — depth-aware
# ============================================================
def extract_bracket_templates(wikitext: str) -> list:
    """
    Find every top-level {{...}} block in wikitext that contains bracket
    match data (identified by 'RD1-team' style params).  Handles nested
    {{ }} and [[ ]] correctly via depth tracking.

    Grand Slam pages typically have 5 templates:
      - 4 section templates  (16 players each, R1 → QF, zero-padded positions)
      - 1 QF/SF/Final template (unpadded positions)
    Olympics pages typically have 3 templates.
    """
    templates = []
    i = 0
    n = len(wikitext)

    while i < n - 1:
        if wikitext[i : i + 2] != "{{":
            i += 1
            continue

        # Scan forward to find the matching }}
        depth = 1
        j = i + 2
        while j < n - 1 and depth > 0:
            if wikitext[j : j + 2] == "{{":
                depth += 1
                j += 2
            elif wikitext[j : j + 2] == "}}":
                depth -= 1
                j += 2
            else:
                j += 1

        template_text = wikitext[i:j]

        # Keep only templates that look like bracket draw tables
        if re.search(r"RD\d+-team", template_text):
            templates.append(template_text)

        i = j  # continue from where this template ended

    return templates

In [ ]:
# ============================================================
# CELL 4: Param extraction — depth-aware parser
# ============================================================
def extract_params(template_text: str) -> dict:
    """
    Parse a bracket template into a {key: value} dict.

    Uses a character-by-character depth-aware parser to correctly split
    params at '|' boundaries, ignoring pipes that are inside nested
    {{ }} (e.g. {{flagicon|USA}}) or [[ ]] (e.g. [[Player|Display]]).

    This avoids the classic regex bug where a param value like
        score1-3 = 6 | seed2=8
    would be incorrectly captured as the value of score1-3.
    """
    inner = template_text.strip()
    if inner.startswith("{{"):
        inner = inner[2:]
    if inner.endswith("}}"):
        inner = inner[:-2]

    parts = []
    current = []
    depth_curly = 0
    depth_square = 0

    for ch in inner:
        if ch == "{":
            depth_curly += 1
            current.append(ch)
        elif ch == "}":
            depth_curly -= 1
            current.append(ch)
        elif ch == "[":
            depth_square += 1
            current.append(ch)
        elif ch == "]":
            depth_square -= 1
            current.append(ch)
        elif ch == "|" and depth_curly == 0 and depth_square == 0:
            parts.append("".join(current).strip())
            current = []
        else:
            current.append(ch)

    if current:
        parts.append("".join(current).strip())

    # parts[0] = template name; the rest are key=value pairs
    params = {}
    for part in parts[1:]:
        if "=" in part:
            key, _, value = part.partition("=")
            key = key.strip()
            value = value.strip()
            if key:
                params[key] = value

    return params


def get_template_type(params: dict) -> str:
    """
    'padded'   → section template  (positions like 01, 02 … 16)
    'unpadded' → QF/SF/Final template (positions 1, 2, 3 …)
    """
    for key in params:
        if re.match(r"^RD\d+-team\d{2}$", key):
            return "padded"
    return "unpadded"

In [ ]:
# ============================================================
# CELL 5: Team name & score parsing
# ============================================================
def parse_team(text: str):
    """
    Parse a team cell from the bracket template.
    Returns (players_str, is_winner).

    Winners are marked with '''bold''' wikitext.
    Player names come from [[Page|Display]] links.
    Flag/icon templates {{...}} are stripped.
    """
    if not text or text.strip() in ("", "&nbsp;", "{{bye}}", "BYE"):
        return None, False

    is_winner = "'''" in text

    # Remove bold markers
    text = text.replace("'''", "")

    # Resolve wiki links: [[Page|Display]] → Display, [[Page]] → Page
    def resolve_link(m):
        inner = m.group(1)
        return inner.split("|")[-1] if "|" in inner else inner

    text = re.sub(r"\[\[([^\]]+)\]\]", resolve_link, text)

    # Remove {{...}} templates (flag icons, nationality templates, etc.)
    # Use depth tracking to handle nested templates
    def strip_templates(s: str) -> str:
        result = []
        depth = 0
        i = 0
        while i < len(s):
            if s[i : i + 2] == "{{":
                depth += 1
                i += 2
            elif s[i : i + 2] == "}}" and depth > 0:
                depth -= 1
                i += 2
            elif depth == 0:
                result.append(s[i])
                i += 1
            else:
                i += 1
        return "".join(result)

    text = strip_templates(text)

    # Remove seedings like (1), [1], or leading numbers
    text = re.sub(r"[\(\[]\s*\d+\s*[\)\]]", "", text)

    # Normalise whitespace
    text = re.sub(r"\s+", " ", text).strip().strip("/").strip()

    return (text if text else None), is_winner


def parse_score_cell(score_text: str):
    """
    Parse one team's score in one set.
    Returns (game_score: int|None, tiebreak: int|None, special: str|None).

    Wikipedia encodes each team's OWN tiebreak points as <sup>X</sup>.
    A match-tiebreak / super-tiebreak may appear as [10] instead.
    special is 'walkover' | 'retired' | 'abandoned' | None.
    """
    if not score_text or score_text.strip() in ("", "&nbsp;"):
        return None, None, None

    t = score_text.strip()

    if re.search(r"w/?o|walkover", t, re.IGNORECASE):
        return None, None, "walkover"
    if re.search(r"ret\.?|retired", t, re.IGNORECASE):
        return None, None, "retired"
    if re.search(r"abd\.?|abandoned", t, re.IGNORECASE):
        return None, None, "abandoned"

    # Tiebreak in <sup>...</sup> (regular set tiebreak)
    tb = None
    sup_m = re.search(r"<sup[^>]*>(\d+)</sup>", t)
    if sup_m:
        tb = int(sup_m.group(1))

    # Tiebreak in [N] (super / match tiebreak format)
    if tb is None:
        brk_m = re.search(r"\[(\d+)\]", t)
        if brk_m:
            tb = int(brk_m.group(1))

    # Main game score: first standalone integer
    game_m = re.search(r"\b(\d+)\b", t)
    game = int(game_m.group(1)) if game_m else None

    return game, tb, None


def build_set_score(g1, tb1, g2, tb2):
    """
    Build a standard-notation set score string.

    Standard tennis notation: the number in parentheses is the SET LOSER's
    tiebreak points.  Wikipedia stores each team's OWN points in <sup>.

    So:
      g1 > g2  → team1 won the set → loser is team2 → show tb2 → '7-6(tb2)'
      g2 > g1  → team2 won the set → loser is team1 → show tb1 → '6-7(tb1)'
    """
    if g1 is None or g2 is None:
        return None

    tb_str = ""
    if tb1 is not None or tb2 is not None:
        tb_val = tb2 if g1 > g2 else tb1
        if tb_val is not None:
            tb_str = f"({tb_val})"

    return f"{g1}-{g2}{tb_str}"


def extract_tb_value(g1, tb1, g2, tb2):
    """
    Return the set loser's tiebreak points as an integer (None if no tiebreak).
    This is the value shown in parentheses in standard notation.
    """
    if tb1 is None and tb2 is None:
        return None
    # Guard: if either game score is missing we cannot determine the set winner
    if g1 is None or g2 is None:
        # Return whichever tb value is present (best-effort)
        return tb1 if tb1 is not None else tb2
    return tb2 if g1 > g2 else tb1

In [ ]:
# ============================================================
# CELL 6: Template → list of match records
# ============================================================
def template_to_matches(params: dict, tournament: str, year: str) -> list:
    """
    Convert the extracted {key: value} params of one bracket template
    into a list of match record dicts.

    Each record contains:
      Tournament, Year, Stage, Round_Order,
      Team1, Seed1, Team2, Seed2, Winner,
      Retired_Walkover,
      Score_Set1…5  (e.g. '7-6(3)'),
      TB_Set1…5    (tiebreak loser points as int, or None).
    """
    ttype = get_template_type(params)

    # Collect round definitions: RD1, RD2, …
    rounds = {}
    for key, val in params.items():
        m = re.match(r"^(RD\d+)$", key)
        if m:
            rounds[m.group(1)] = val

    matches = []

    for rd_key in sorted(rounds.keys()):
        stage_raw = rounds[rd_key]
        stage = STAGE_NORMALIZE.get(stage_raw.lower().strip(), stage_raw.strip())

        # Build regex pattern to find all team positions in this round
        if ttype == "padded":
            pos_pattern = rf"^{rd_key}-team(\d{{2}})$"
        else:
            pos_pattern = rf"^{rd_key}-team(\d+)$"

        positions = sorted(
            [re.match(pos_pattern, k).group(1) for k in params if re.match(pos_pattern, k)],
            key=lambda x: int(x),
        )

        # Consecutive pairs of positions = one match
        for i in range(0, len(positions) - 1, 2):
            p1, p2 = positions[i], positions[i + 1]

            t1_text = params.get(f"{rd_key}-team{p1}", "")
            t2_text = params.get(f"{rd_key}-team{p2}", "")

            team1, t1_winner = parse_team(t1_text)
            team2, t2_winner = parse_team(t2_text)

            # Skip byes / empty slots
            if team1 is None and team2 is None:
                continue

            # Seeds are stored in dedicated params (separate from team name text)
            raw_s1 = params.get(f"{rd_key}-seed{p1}", "").strip()
            raw_s2 = params.get(f"{rd_key}-seed{p2}", "").strip()

            def _parse_seed(s):
                """Return int if numeric seed, otherwise raw string (WC, Q, LL…)."""
                if not s:
                    return None
                # Strip wikitext if any
                s = re.sub(r"\{\{.*?\}\}", "", s)
                s = re.sub(r"\[\[.*?\]\]", "", s).strip()
                try:
                    return int(s)
                except ValueError:
                    return s or None

            seed1 = _parse_seed(raw_s1)
            seed2 = _parse_seed(raw_s2)

            winner = team1 if t1_winner else (team2 if t2_winner else None)

            # Parse set scores (up to 5 sets)
            set_scores = []
            set_tbs    = []
            is_special = None

            for set_num in range(1, 6):
                s1_key = f"{rd_key}-score{p1}-{set_num}"
                s2_key = f"{rd_key}-score{p2}-{set_num}"
                s1_text = params.get(s1_key, "")
                s2_text = params.get(s2_key, "")

                if not s1_text and not s2_text:
                    break

                g1, tb1, sp1 = parse_score_cell(s1_text)
                g2, tb2, sp2 = parse_score_cell(s2_text)

                if sp1 or sp2:
                    is_special = sp1 or sp2
                    break

                score_str = build_set_score(g1, tb1, g2, tb2)
                tb_val    = extract_tb_value(g1, tb1, g2, tb2)

                set_scores.append(score_str)
                set_tbs.append(tb_val)

            record = {
                "Tournament":       tournament,
                "Year":             year,
                "Stage":            stage,
                "Round_Order":      ROUND_ORDER.get(stage, 99),
                "Team1":            team1,
                "Seed1":            seed1,
                "Team2":            team2,
                "Seed2":            seed2,
                "Winner":           winner,
                "Retired_Walkover": is_special,
            }

            for idx, (sc, tb) in enumerate(zip(set_scores, set_tbs), 1):
                record[f"Score_Set{idx}"] = sc
                record[f"TB_Set{idx}"]    = tb

            matches.append(record)

    return matches

In [ ]:
# ============================================================
# CELL 7: Tournament scraper
# ============================================================
def scrape_tournament(page_title: str, tournament: str, year: str) -> list:
    """
    Full pipeline for one tournament page:
      fetch wikitext → extract templates → parse params → extract matches.
    Returns a list of match record dicts.
    """
    print(f"Fetching: {page_title}")

    try:
        wikitext = fetch_wikitext(page_title)
    except Exception as e:
        print(f"  [ERROR] Could not fetch page: {e}")
        return []

    templates = extract_bracket_templates(wikitext)
    print(f"  Found {len(templates)} bracket template(s)")

    all_matches = []
    for tmpl in templates:
        params = extract_params(tmpl)
        matches = template_to_matches(params, tournament, year)
        all_matches.extend(matches)

    # Deduplicate: the same match sometimes appears in both a section
    # template (as the QF) and the QF/SF/Final template.
    seen  = set()
    unique = []
    for m in all_matches:
        key = (
            m["Tournament"], m["Year"], m["Stage"],
            m.get("Team1") or "",
            m.get("Team2") or "",
        )
        if key not in seen:
            seen.add(key)
            unique.append(m)

    print(f"  Extracted {len(unique)} unique matches")
    return unique

In [ ]:
# ============================================================
# CELL 8: Quick diagnostic — test one tournament before full run
# ============================================================
test_page, test_tourn, test_year = TOURNAMENTS[0]   # 2024 Australian Open

test_records = scrape_tournament(test_page, test_tourn, test_year)
test_df = pd.DataFrame(test_records)

print(f"\nColumns: {list(test_df.columns)}")
print(f"Stages found: {test_df['Stage'].value_counts().to_dict()}")
print()

# Show matches that had at least one tiebreak
tb_cols = [c for c in test_df.columns if c.startswith("TB_")]
has_tb = test_df[tb_cols].notna().any(axis=1)
print(f"Matches with tiebreaks: {has_tb.sum()} / {len(test_df)}")
test_df[has_tb][["Stage", "Team1", "Team2", "Winner"] + [c for c in test_df.columns if c.startswith("Score_") or c.startswith("TB_")]].head(10)

In [ ]:
# ============================================================
# CELL 9: Full run — all tournaments
# ============================================================
all_records = []

for page_title, tournament, year in TOURNAMENTS:
    records = scrape_tournament(page_title, tournament, year)
    all_records.extend(records)
    time.sleep(DELAY)   # be polite to Wikipedia

df = pd.DataFrame(all_records)

# Sort: year → tournament → round
df = df.sort_values(["Year", "Tournament", "Round_Order"]).reset_index(drop=True)

# Summary
print(f"\n{'='*55}")
print(f"Total matches collected: {len(df)}")
print()
summary = df.groupby(["Year", "Tournament"]).size().rename("Matches")
print(summary.to_string())
print()

# Tiebreak summary
tb_cols = [c for c in df.columns if c.startswith("TB_")]
tb_count = df[tb_cols].notna().sum()
print("Tiebreak counts per set column:")
print(tb_count[tb_count > 0].to_string())

df.head()

In [ ]:
# ============================================================
# CELL 10: Save to Excel
# ============================================================
out_path = os.path.join(OUTPUT_DIR, "grand_slam_matches_2024_2025_olympics.xlsx")
df.to_excel(out_path, index=False)
print(f"Saved {len(df)} rows → {out_path}")